In [0]:
# ======================================
# Dataset: olist_order_items
# Layer: Silver
# Source: Bronze (delta)
# Grain: order_id + order_item_id
# ======================================


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window


In [0]:
%run ../commons/silver_utils

In [0]:
%run ../../00_config/paths

In [0]:
df_bronze = spark.read.format('delta').load(BRONZE_ORDER_ITEMS_PATH)
df_bronze.printSchema()

In [0]:
df = normalize_columns(df_bronze)

In [0]:
df = df.select(
    F.col("order_id"),
    F.col("product_id"),
    F.col("seller_id"),
    F.col("order_item_id").cast(IntegerType()).alias("order_item_id"),
    F.to_timestamp(F.col("shipping_limit_date"), "yyyy-MM-dd HH:mm:ss").alias("shipping_limit_date"),
    F.col("price").cast(DecimalType(10, 2)).alias("price"),
    F.col("freight_value").cast(DecimalType(10, 2)).alias("freight_value")
)

In [0]:
df.printSchema()

In [0]:
df = df.filter(
    F.col("order_id").isNotNull() &
    F.col("order_item_id").isNotNull() &
    F.col("product_id").isNotNull() &
    F.col("seller_id").isNotNull()
)


In [0]:
df = df.filter(
    (F.col("price") > 0) &
    (F.col("freight_value") >= 0)
)


In [0]:
df.printSchema()

In [0]:
print("Bronze count:", df_bronze.count())
print("Silver count:", df.count())


In [0]:
write_silver(df, SILVER_ORDER_ITEMS_PATH)
